# Build-up vs press: annotated clips + animation
Run **notebook 1 first** (it writes `buildup_analysis/*.csv`). This notebook auto-selects the most informative build-up phases (press wins the ball, press is beaten, goalkeeper under press, man-oriented high press, free build-up) and renders each as an mp4:

- **left:** your `annotated_teams.mp4` with the press layer on top - presser rings + lines to the carrier, green/red passing lanes (open vs cut), carrier triangle, HUD;
- **right:** an animated top-down view in the building team's frame (opponent hull + press line, ball trail, pass arrows, pressers-over-time timeline).

Output: `buildup_analysis/clips/*.mp4` plus a preview jpg per clip.

## 1. Setup & load

In [ ]:
# ===== 1. Setup, parameters, load (same parameters as notebook 1 - keep them in sync) =====
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from scipy.optimize import linear_sum_assignment
import warnings; warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)

CACHE = Path(r"C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)"
             r"\project\barca_atletico_first_half\new_cache")
PLAYER_PATH = CACHE / "player_frame_table_named.parquet"
BALL_PATH = CACHE / "ball_frame_table_named.parquet"
OUT_DIR = CACHE / "buildup_analysis"            # written by notebook 1 (cell 9)
VIDEO_PATH = CACHE / "annotated_teams.mp4"
CLIP_DIR = OUT_DIR / "clips"

# ---------------- parameters (all tunable) ----------------
L, W = 105.0, 68.0
# --- what counts as a build-up phase (all coordinates in the BUILDING team's frame: own goal x=0, attacking +x)
BUILDUP_START_X = 40.0    # possession must begin with the ball at x <= this
BUILDUP_END_X   = 52.5    # phase ends when the ball crosses this line while still in possession
MIN_PHASE_SEC   = 1.5
POSS_GAP_SEC    = 4.0     # fill possession_team gaps up to this long if both sides are the same team
CARRIER_GAP_SEC = 0.6     # fill carrier gaps up to this long if the same player is on both sides
BALL_INTERP_SEC = 1.5
LOOKBACK_REGAIN_SEC = 3.0 # opponent had the ball within this window before the phase -> "regain"
# --- passes
MIN_RUN_FRAMES  = 2
MAX_PASS_GAP_SEC= 2.5
FWD_DX, PROG_DX, LONG_PASS_M = 3.0, 10.0, 25.0
# --- press measurement
SAMPLE_SEC      = 0.5
PRESS_R, CLOSE_R, CLOSING_SPEED, CLOSING_LAG_SEC = 8.0, 3.5, 2.0, 0.5
OPTION_MIN, OPTION_MAX = 5.0, 35.0    # passing options = teammates this far from the ball
MARK_R, LANE_BLOCK_R   = 3.5, 2.0     # option is "cut" if marked within MARK_R or an opponent sits on the passing lane
HIGH_PRESS_FRONT3, MID_BLOCK_FRONT3 = 65.0, 50.0   # opponent front-3 height (m from THEIR goal)
WINDOW_SAMPLES, SHADOW_MAN, SHADOW_ZONAL = 8, 0.55, 0.30
PRESSED_PHASE_SHARE = 0.30            # phase counts as "pressed" if >=30% of samples have a presser
FLIP_LANES = False                    # set True if "left/right" come out mirrored
MIN_TRACKED = 7

# ---------------- load ----------------
need = ["frame_idx","track_id","role","team","team_name","name","number","pitch_x","pitch_y","t_sec",
        "attack_dir","x_att","carrier_track_id","possession_team","is_stoppage","is_carrier",
        "x1","y1","x2","y2","foot_px_x","foot_px_y"]
P  = pd.read_parquet(PLAYER_PATH, columns=need)
Bt = pd.read_parquet(BALL_PATH)
print("player rows:", len(P), "| frames:", P.frame_idx.nunique())
print("roles:", P.role.value_counts(dropna=False).to_dict())
print("\nBALL TABLE columns:", list(Bt.columns))
print(Bt.describe().T[["count","min","max","mean"]].head(25))

# ---------------- clip settings ----------------
N_PER_CATEGORY   = 1        # clips per team per category (5 categories) - raise once you like the output
MANUAL_PHASES    = []       # e.g. [12, 45] to force specific phase_id values from buildup_phases.csv
PRE_ROLL_SEC, POST_ROLL_SEC, MAX_CLIP_SEC = 2.0, 1.5, 22.0
RENDER_MODE      = "both"   # "both" (video overlay + top-down animation) | "overlay" | "tactical"
OUT_STRIDE       = 2        # 2 = write every 2nd frame (12.5 fps, ~2x faster); 1 = full frame rate
TRACK_W, TRACK_H = 1920, 1080   # pixel space of x1,y1,x2,y2,foot_px_* (the resolution tracking ran on)
VIDEO_FRAME_OFFSET = 0      # video frame = frame_idx + offset (use if the mp4 is trimmed differently)
DRAW_NUMBERS     = False    # annotated_teams.mp4 is probably already labelled; True adds small shirt numbers


## 2. Prep (same as notebook 1)

In [ ]:
# ===== 2. Preparation: teams, orientation, frame table, ball track =====
P = P.drop_duplicates(["frame_idx", "track_id"]).copy()
for c in ["team", "possession_team", "carrier_track_id", "number"]:
    P[c] = pd.to_numeric(P[c], errors="coerce").astype("float64")
P["is_stoppage"] = P["is_stoppage"].fillna(False).astype(bool)
P["is_carrier"]  = P["is_carrier"].fillna(False).astype(bool)

FMAX = int(P.frame_idx.max())
fps  = float(round((P.frame_idx.max() - P.frame_idx.min()) / (P.t_sec.max() - P.t_sec.min()), 2))
sec  = lambda s: max(1, int(round(s * fps)))
print(f"fps = {fps} | frames 0..{FMAX} | {FMAX/fps/60:.1f} min")

rl = P["role"].astype(str).str.lower()
P["is_ref"] = rl.str.contains("ref")
P["is_gk"]  = rl.str.startswith("goal") | rl.eq("gk")
P["is_out"] = P.team.notna() & ~P.is_gk & ~P.is_ref        # outfield players only

tn = P.dropna(subset=["team"]).groupby("team")["team_name"].agg(lambda s: s.mode().iat[0])
TEAM_NAME = {int(k): v for k, v in tn.items()}
TEAMS = sorted(TEAM_NAME)
ATT_DIR = {int(k): float(np.sign(v)) for k, v in
           P.dropna(subset=["team", "attack_dir"]).groupby("team")["attack_dir"].mean().items()}
print("teams:", TEAM_NAME, "| attack direction (+1 = towards x=105):", ATT_DIR)
assert all(v in (1.0, -1.0) for v in ATT_DIR.values()) and len(set(ATT_DIR.values())) == 2, \
    "attack_dir is not constant per team (first half should be) - check the table"

def to_frame(x, y, team):
    # absolute pitch coords -> frame in which `team` attacks towards +x (own goal at x=0)
    if ATT_DIR[int(team)] == 1.0:
        return x, y
    return L - x, W - y

# sanity: my own-frame x vs the pipeline's x_att
_d = P["team"].map(ATT_DIR)
_xo = np.where(_d == 1.0, P.pitch_x, L - P.pitch_x)
print("median |x_own - x_att| (should be ~0):", float(np.nanmedian(np.abs(_xo - P.x_att))))

# ---- track info (names are unique per identity in the named cache)
def mode_or_nan(s):
    s = s.dropna()
    return s.mode().iat[0] if len(s) else np.nan
info = P.groupby("track_id").agg(name=("name", mode_or_nan), number=("number", mode_or_nan),
                                 team=("team", mode_or_nan), is_gk=("is_gk", "max"), is_ref=("is_ref", "max"))
info.index = info.index.astype(int)
info["label"] = [(f"#{int(n)} {nm}" if pd.notna(n) else str(nm)) if pd.notna(nm) else f"id{i}"
                 for i, n, nm in zip(info.index, info.number, info.name)]
TEAM_OF  = info["team"].to_dict(); LABEL = info["label"].to_dict(); GK_IDS = set(info.index[info.is_gk])

# ---- one row per frame: possession / carrier (before dropping rows without pitch coords)
fr = (P.groupby("frame_idx").agg(possession_team=("possession_team", "first"),
                                 carrier=("carrier_track_id", "first"),
                                 stop=("is_stoppage", "max")).reindex(np.arange(FMAX + 1)))
fr["stop"] = fr["stop"].fillna(False).astype(bool)

def fill_short_gaps(s, max_len):
    # fill NaN runs <= max_len only when the value before and after the gap is identical
    isn = s.isna()
    grp = (isn != isn.shift()).cumsum()
    run_len = isn.groupby(grp).transform("sum")
    f, b = s.ffill(), s.bfill()
    m = isn & (run_len <= max_len) & (f == b)
    out = s.copy(); out[m] = f[m]
    return out

car_f = fill_short_gaps(fr["carrier"].where(~fr["stop"]), sec(CARRIER_GAP_SEC))
poss0 = fr["possession_team"].where(~fr["stop"])
poss0 = poss0.fillna(car_f.map(lambda v: TEAM_OF.get(int(v), np.nan) if pd.notna(v) else np.nan))
poss_f = fill_short_gaps(poss0, sec(POSS_GAP_SEC))
print(f"possession known: raw {fr.possession_team.notna().mean():.1%} -> after gap fill {poss_f.notna().mean():.1%} | "
      f"carrier known: raw {fr.carrier.notna().mean():.1%} -> {car_f.notna().mean():.1%}")

# ---- now keep rows with pitch coordinates
P["px"] = P.pitch_x.clip(0, L); P["py"] = P.pitch_y.clip(0, W)
P = P[P.px.notna() & P.py.notna()].copy()

# ---- ball position per frame (ball table first, carrier foot as fallback)
cands_x = ["pitch_x", "ball_pitch_x", "ball_x", "x_pitch"]; cands_y = ["pitch_y", "ball_pitch_y", "ball_y", "y_pitch"]
BALL_X_COL = next((c for c in cands_x if c in Bt.columns), None)    # override here if the wrong one is picked
BALL_Y_COL = next((c for c in cands_y if c in Bt.columns), None)
bpos = pd.DataFrame(index=np.arange(FMAX + 1), columns=["x", "y"], dtype="float64")
if BALL_X_COL and BALL_Y_COL:
    bb = Bt.drop_duplicates("frame_idx").set_index("frame_idx")[[BALL_X_COL, BALL_Y_COL]]
    bb.columns = ["x", "y"]; bpos.loc[bb.index.intersection(bpos.index)] = bb.loc[bb.index.intersection(bpos.index)]
    print(f"ball coords from ball table ({BALL_X_COL}, {BALL_Y_COL}): {bpos.x.notna().mean():.1%} of frames")
cp = P[P.is_carrier].groupby("frame_idx")[["px", "py"]].first()
bpos["x"] = bpos["x"].fillna(cp["px"]); bpos["y"] = bpos["y"].fillna(cp["py"])
bpos = bpos.interpolate(limit=sec(BALL_INTERP_SEC), limit_area="inside")
bpos["x"] = bpos["x"].clip(0, L); bpos["y"] = bpos["y"].clip(0, W)
ball_x, ball_y = bpos["x"].values, bpos["y"].values
print(f"ball position available (after carrier fallback + interpolation): {bpos.x.notna().mean():.1%}")


## 3. Auto-select clips

In [ ]:
# ===== 3. Load build-up phases from notebook 1 + auto-select clips =====
PH = pd.read_csv(OUT_DIR / "buildup_phases.csv"); PASS_PH = pd.read_csv(OUT_DIR / "buildup_passes.csv")
PH["pressed"] = PH.pressed.map(lambda v: str(v) == "True")
PH["press_intensity"] = PH.press_share.fillna(0) * PH.n_press_mean.fillna(0)
print(f"{len(PH)} phases loaded | video: {VIDEO_PATH.name} exists={VIDEO_PATH.exists()}")

CATS = {   # name -> (filter, score used to rank)
    "press_wins_ball":         (lambda d: d[(d.outcome == "lost") & d.pressed & (d.dur >= 3)], "press_intensity"),
    "escapes_press":           (lambda d: d[(d.outcome == "progressed") & d.pressed & (d.n_passes >= 2)], "press_intensity"),
    "goalkeeper_vs_press":     (lambda d: d[(d.start_type == "goalkeeper") & d.pressed], "press_intensity"),
    "man_oriented_high_press": (lambda d: d[(d.press_style == "man-oriented") & (d.press_block == "high press")], "press_intensity"),
    "free_build_up":           (lambda d: d[(~d.pressed) & (d.outcome == "progressed") & (d.n_passes >= 4)], "n_passes"),
}
chosen = []
for t in TEAMS:
    dt, used = PH[PH.team == t], set()
    for cat, (flt, score) in CATS.items():
        cand = flt(dt); cand = cand[~cand.phase_id.isin(used)].sort_values(score, ascending=False).head(N_PER_CATEGORY)
        for r in cand.itertuples(): chosen.append((r.phase_id, cat)); used.add(r.phase_id)
for pid in MANUAL_PHASES: chosen.append((pid, "manual"))
if not chosen:   # fallback so you always get something to look at
    for t in TEAMS:
        for r in PH[PH.team == t].sort_values("press_intensity", ascending=False).head(2).itertuples(): chosen.append((r.phase_id, "most_pressed"))

CL = pd.DataFrame(chosen, columns=["phase_id", "category"]).merge(PH, on="phase_id")
CL["fs"] = (CL.f0 - sec(PRE_ROLL_SEC)).clip(lower=0).astype(int)
CL["fe"] = (np.minimum(CL.f1, CL.f0 + sec(MAX_CLIP_SEC)) + sec(POST_ROLL_SEC)).clip(upper=FMAX).astype(int)
CL["clock"] = CL.t0.map(lambda t: f"{int(t // 60):02d}:{int(t % 60):02d}")
CL["name"] = [f"{TEAM_NAME[int(r.team)]}_{r.category}_ph{int(r.phase_id)}_{r.clock.replace(':', 'm')}s" for r in CL.itertuples()]
print(CL[["name", "clock", "dur", "outcome", "start_type", "n_passes", "press_block", "press_style", "press_share", "n_press_mean"]].round(2).to_string(index=False))


## 4. Per-frame press engine

In [ ]:
# ===== 4. Per-frame press engine (same definitions as notebook 1, but at every video frame) =====
from scipy.spatial import ConvexHull
car_arr = car_f.values
def pt_seg_dist(pts, a, b):
    ab = b - a; ap = pts[:, None, :] - a
    t = np.clip((ap * ab[None]).sum(-1) / np.maximum((ab ** 2).sum(-1)[None], 1e-9), 0, 1)
    return np.linalg.norm(pts[:, None, :] - (a + t[..., None] * ab[None]), axis=-1)
def block_of(v):
    if v is None or pd.isna(v): return "n/a"
    return "high press" if v >= HIGH_PRESS_FRONT3 else ("mid block" if v >= MID_BLOCK_FRONT3 else "low block")

def frame_state(g, f, A):
    st = dict(f=f, rows=g.set_index("track_id"))
    st["ball"] = None if np.isnan(ball_x[f]) else np.array(to_frame(ball_x[f], ball_y[f], A))
    gA, gB = g[g.team == A], g[g.team != A]
    ax_, ay_ = to_frame(gA.px.values, gA.py.values, A); bx_, by_ = to_frame(gB.px.values, gB.py.values, A)
    st.update(A_ids=gA.track_id.values.astype(int), A_xy=np.c_[ax_, ay_], A_gk=gA.is_gk.values,
              B_ids=gB.track_id.values.astype(int), B_xy=np.c_[bx_, by_], B_gk=gB.is_gk.values, B_out=gB.is_out.values)
    car = car_arr[f]; st["car"] = None if np.isnan(car) else int(car)
    Bo = gB[gB.is_out]
    pm = ((Bo.d_ball <= CLOSE_R) | ((Bo.d_ball <= PRESS_R) & (Bo.closing.fillna(-9) >= CLOSING_SPEED))).values
    st["pressers"] = Bo.track_id.values[pm].astype(int); st["n_press"] = int(pm.sum())
    bOwn = L - st["B_xy"][st["B_out"], 0]
    st["front3"] = float(np.sort(bOwn)[-3:].mean()) if len(bOwn) >= MIN_TRACKED else None
    st["opt_ids"], st["opt_cut"] = np.array([], int), np.array([], bool)
    if st["ball"] is not None and len(Bo):
        keep = st["A_ids"] != (st["car"] if st["car"] is not None else -1)
        oxy, oid = st["A_xy"][keep], st["A_ids"][keep]
        dd = np.linalg.norm(oxy - st["ball"], axis=1); sel = (dd >= OPTION_MIN) & (dd <= OPTION_MAX)
        if sel.any():
            Bxy = st["B_xy"][st["B_out"]]; O = oxy[sel]
            cut = (np.linalg.norm(Bxy[:, None] - O[None], axis=-1).min(0) <= MARK_R) | (pt_seg_dist(Bxy, st["ball"], O) <= LANE_BLOCK_R).any(0)
            st["opt_ids"], st["opt_cut"] = oid[sel], cut
    return st

def clip_state(fs, fe, A):
    lagf = sec(CLOSING_LAG_SEC)
    Pc = P[(P.frame_idx >= fs - lagf) & (P.frame_idx <= fe) & P.team.notna()].sort_values(["track_id", "frame_idx"]).copy()
    Pc["d_ball"] = np.hypot(Pc.px.values - ball_x[Pc.frame_idx.values], Pc.py.values - ball_y[Pc.frame_idx.values])
    gb = Pc.groupby("track_id")
    Pc["closing"] = np.where((Pc.frame_idx - gb["frame_idx"].shift(lagf)) == lagf, (gb["d_ball"].shift(lagf) - Pc.d_ball) / (lagf / fps), np.nan)
    Pc = Pc[Pc.frame_idx >= fs]
    return {int(f): frame_state(g, int(f), A) for f, g in Pc.groupby("frame_idx")}

# ---- ball in video pixels (real detections only, as in your annotation style)
BALL_PX_PAIRS = [("ball_px_x", "ball_px_y"), ("px_x", "px_y"), ("cx", "cy"), ("ball_cx", "ball_cy"), ("x_px", "y_px"),
                 ("ball_x_px", "ball_y_px"), ("foot_px_x", "foot_px_y"), ("x", "y")]
BALL_PX = None
for xc, yc in BALL_PX_PAIRS:
    if xc in Bt.columns and yc in Bt.columns and Bt[xc].quantile(.95) > 200: BALL_PX = (xc, yc); break
BALL_PX_MAP = {}
if BALL_PX:
    bq = Bt.drop_duplicates("frame_idx").copy()
    for dc in ["detected", "is_detected", "ball_detected"]:
        if dc in bq.columns: bq = bq[bq[dc].astype(bool)]; print("ball detections filtered with", dc); break
    else:
        if "source" in bq.columns:
            print("ball table 'source' values:", bq.source.value_counts().to_dict(), "-> keeping those that look like detections")
            bq = bq[bq.source.astype(str).str.lower().isin(["detected", "detection", "det", "yolo", "measured"])]
    BALL_PX_MAP = bq.dropna(subset=list(BALL_PX)).set_index("frame_idx")[list(BALL_PX)].T.to_dict("list")
print("ball pixel columns:", BALL_PX, "| detected-ball frames available:", len(BALL_PX_MAP))
NUM = {int(i): int(n) for i, n in info.number.dropna().items()}


## 5. Renderers

In [ ]:
# ===== 5. Renderers: (a) press overlay on the video, (b) top-down animated panel, (c) side-by-side composite =====
import cv2, shutil, subprocess, time
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
BGR = dict(orange=(0, 150, 255), green=(80, 220, 80), red=(60, 60, 235), yellow=(0, 230, 255), white=(255, 255, 255))

def dashed(img, p0, p1, color, th, dash=14):
    p0, p1 = np.array(p0, float), np.array(p1, float); n = int(np.linalg.norm(p1 - p0) // dash)
    for i in range(0, n, 2):
        a = p0 + (p1 - p0) * i / n; b = p0 + (p1 - p0) * min(i + 1, n) / n
        cv2.line(img, tuple(a.astype(int)), tuple(b.astype(int)), color, th, cv2.LINE_AA)

def pix(st, tid):
    # (foot_x, foot_y, x1, y1, x2, y2) in tracking-pixel space, or None
    if tid is None or tid not in st["rows"].index: return None
    r = st["rows"].loc[tid]
    v = [r.foot_px_x, r.foot_px_y, r.x1, r.y1, r.x2, r.y2]
    return None if any(pd.isna(v)) else v

def draw_overlay(img, st, f, meta, ph, pidx, n_ph_pass):
    h, w = img.shape[:2]; sx, sy = w / TRACK_W, h / TRACK_H; th = max(1, int(round(2 * w / 1920)))
    S = lambda p: (int(p[0] * sx), int(p[1] * sy))
    in_phase = ph.f0 <= f <= ph.f1
    if st is not None:
        car = pix(st, st["car"])
        # passing options: green = open, red dashed = cut by the press
        if car is not None:
            for oid, cut in zip(st["opt_ids"], st["opt_cut"]):
                o = pix(st, int(oid))
                if o is None: continue
                if cut: dashed(img, S(car[:2]), S(o[:2]), BGR["red"], th)
                else:   cv2.line(img, S(car[:2]), S(o[:2]), BGR["green"], th, cv2.LINE_AA)
        # pressers: ring at the feet + line to the carrier
        for pid_ in st["pressers"]:
            p = pix(st, int(pid_))
            if p is None: continue
            cv2.ellipse(img, S(p[:2]), (int(26 * sx), int(9 * sy)), 0, 0, 360, BGR["orange"], th + 1, cv2.LINE_AA)
            if car is not None: cv2.line(img, S(p[:2]), S(car[:2]), BGR["orange"], max(1, th - 1), cv2.LINE_AA)
        # carrier: small flipped triangle fixed above the head
        if car is not None:
            cx, top = int((car[2] + car[4]) / 2 * sx), int(car[3] * sy)
            tri = np.array([[cx, top - 4 * sy], [cx - 9 * sx, top - 22 * sy], [cx + 9 * sx, top - 22 * sy]], np.int32)
            cv2.fillPoly(img, [tri], BGR["yellow"]); cv2.polylines(img, [tri], True, (0, 0, 0), 1, cv2.LINE_AA)
        if DRAW_NUMBERS:
            for tid in st["rows"].index:
                p = pix(st, tid); n = NUM.get(int(tid))
                if p is not None and n is not None and not st["rows"].loc[tid].is_ref:
                    cv2.putText(img, str(n), (int((p[2] + p[4]) / 2 * sx) - 6, int(p[3] * sy) - 26), cv2.FONT_HERSHEY_SIMPLEX, max(0.35, 0.6 * w / 1920), BGR["white"], 1, cv2.LINE_AA)
    if f in BALL_PX_MAP:
        bx, by = BALL_PX_MAP[f]; cv2.circle(img, (int(bx * sx), int(by * sy)), max(4, int(7 * sx)), BGR["yellow"], -1, cv2.LINE_AA)
    # HUD
    lines = [f"{meta['A']} build-up vs {meta['B']} press  |  {meta['clock']}  |  {meta['category']}",
             (f"pressers {st['n_press']}  options cut {int(st['opt_cut'].sum())}/{len(st['opt_cut'])}  block {block_of(st['front3'])}" if st is not None else ""),
             "BUILD-UP IN PLAY" if in_phase else ("context (before phase)" if f < ph.f0 else f"phase over: {ph.outcome}")]
    sc = w / 1920; pad = max(6, int(16 * sc)); lh = max(14, int(40 * sc)); fs_ = max(0.45, 0.9 * sc)
    x0, y0, x1, y1 = pad, pad, pad + int(1150 * sc), pad + lh * 3 + pad // 2
    roi = img[y0:y1, x0:x1]; img[y0:y1, x0:x1] = cv2.addWeighted(roi, 0.35, np.zeros_like(roi), 0.65, 0)
    for i, t in enumerate(lines):
        cv2.putText(img, t, (x0 + pad // 2, y0 + lh * (i + 1) - pad // 3), cv2.FONT_HERSHEY_SIMPLEX, fs_, BGR["white"], max(1, int(2 * sc)), cv2.LINE_AA)
    return img

PANEL_W, PANEL_H = 640, 720
def draw_pitch_panel(ax):
    ax.set_facecolor("#2d6a30"); ax.set_xlim(-2, L + 2); ax.set_ylim(W + 2, -2); ax.set_aspect("equal"); ax.axis("off")
    c = "#d8d8d8"; ax.add_patch(Rectangle((-2, -2), L + 4, W + 4, fc="#2d6a30", ec="none", zorder=0))
    ax.add_patch(Rectangle((0, 0), L, W, fill=False, ec=c, lw=1)); ax.plot([L / 2] * 2, [0, W], c=c, lw=1)
    ax.add_patch(Circle((L / 2, W / 2), 9.15, fill=False, ec=c, lw=1))
    for x0, w_ in [(0, 16.5), (L - 16.5, 16.5)]: ax.add_patch(Rectangle((x0, W / 2 - 20.16), w_, 40.32, fill=False, ec=c, lw=1))
    ax.text(3, -0.8, "own goal", color=c, fontsize=7, va="bottom"); ax.text(L - 3, -0.8, "attacking >", color=c, fontsize=7, ha="right", va="bottom")

class Panel:
    A_COL, B_COL = "#4da3ff", "#ff9f43"
    def __init__(self, FR, ph, pas, meta):
        self.FR, self.ph, self.pas, self.meta = FR, ph, pas, meta
        self.fig = Figure(figsize=(PANEL_W / 100, PANEL_H / 100), dpi=100, facecolor="#111418"); FigureCanvasAgg(self.fig)
        self.axp = self.fig.add_axes([.02, .47, .96, .5]); draw_pitch_panel(self.axp)
        self.axt = self.fig.add_axes([.04, .17, .92, .28]); self.axt.axis("off"); self.axt.set_xlim(0, 1); self.axt.set_ylim(0, 1)
        self.axl = self.fig.add_axes([.09, .04, .86, .11], facecolor="#1b1f26")
        fl = np.array(sorted(FR)); self.fl = fl; npz = np.array([FR[f]["n_press"] for f in fl])
        self.axl.fill_between(fl, npz, step="post", color=self.B_COL, alpha=.8); self.axl.set_ylim(0, max(3, npz.max() + 1))
        self.axl.axvspan(ph.f0, ph.f1, color="white", alpha=.07)
        for p in pas.itertuples(): self.axl.axvline(p.f_rel, c="white", lw=.8, alpha=.7)
        self.axl.set_xlim(fl.min(), fl.max()); self.axl.tick_params(colors="#aaa", labelsize=7)
        self.axl.set_xticks([]); self.axl.set_ylabel("pressers", color="#aaa", fontsize=7)
        self.cursor = self.axl.axvline(fl.min(), c="#ff4d4d", lw=1.5); self.dyn = []
    def _add(self, a): self.dyn.extend(a if isinstance(a, (list, tuple)) else [a])
    def render(self, f):
        for a in self.dyn: a.remove()
        self.dyn = []; ax = self.axp; st = self.FR.get(f); ph = self.ph; self.cursor.set_xdata([f, f])
        if st is not None:
            Bo = st["B_xy"][st["B_out"]]
            if len(Bo) >= 3:
                poly = Bo[ConvexHull(Bo).vertices]; self._add(ax.fill(poly[:, 0], poly[:, 1], color=self.B_COL, alpha=.13, zorder=1))
            if st["front3"] is not None:
                self._add(ax.plot([L - st["front3"]] * 2, [0, W], c=self.B_COL, ls="--", lw=1.4, alpha=.9, zorder=2))
            past = [self.FR[g]["ball"] for g in range(f - sec(3), f + 1) if g in self.FR and self.FR[g]["ball"] is not None]
            if len(past) > 1:
                tr = np.array(past); self._add(ax.plot(tr[:, 0], tr[:, 1], c="#ffe600", lw=1.3, alpha=.55, zorder=2))
            apos = dict(zip(st["A_ids"], st["A_xy"])); src = st["ball"]
            if src is not None:
                for oid, cut in zip(st["opt_ids"], st["opt_cut"]):
                    o = apos.get(int(oid))
                    if o is not None:
                        self._add(ax.plot([src[0], o[0]], [src[1], o[1]], c="#ff4d4d" if cut else "#7cfc00", ls=":" if cut else "-", lw=1.4, alpha=.9, zorder=3))
            self._add(ax.scatter(st["A_xy"][~st["A_gk"], 0], st["A_xy"][~st["A_gk"], 1], s=170, c=self.A_COL, ec="white", lw=1, zorder=4))
            if st["A_gk"].any(): self._add(ax.scatter(*st["A_xy"][st["A_gk"]].T, s=170, c=self.A_COL, ec="white", marker="s", zorder=4))
            self._add(ax.scatter(st["B_xy"][:, 0], st["B_xy"][:, 1], s=170, c=self.B_COL, ec="white", lw=1, zorder=4))
            for tid, xy in list(zip(st["A_ids"], st["A_xy"])) + list(zip(st["B_ids"], st["B_xy"])):
                n = NUM.get(int(tid)); self._add(ax.text(xy[0], xy[1], "" if n is None else str(n), ha="center", va="center", fontsize=6.5, color="white", weight="bold", zorder=5))
            bm = dict(zip(st["B_ids"], st["B_xy"]))
            for pid_ in st["pressers"]:
                self._add(ax.scatter(*bm[int(pid_)], s=420, facecolors="none", edgecolors="#ffe600", lw=1.8, zorder=5))
            if st["car"] is not None and st["car"] in apos:
                self._add(ax.scatter(*apos[st["car"]], s=380, facecolors="none", edgecolors="white", lw=2.2, zorder=5))
            if src is not None: self._add(ax.scatter(*src, s=60, c="#ffe600", ec="black", zorder=6))
        for p in self.pas.itertuples():
            if p.f_rel <= f <= p.f_recv + sec(0.8):
                self._add(ax.annotate("", xy=(p.x1, p.y1), xytext=(p.x0, p.y0), arrowprops=dict(arrowstyle="-|>", color="white", lw=2.4), zorder=7))
        # text block
        m = self.meta; npass = int((self.pas.f_rel <= f).sum()); t = self.axt
        self._add(t.text(0, .96, f"{m['A']} build-up  vs  {m['B']} press", color="white", fontsize=12, weight="bold", va="top"))
        self._add(t.text(0, .80, f"{m['clock']}  |  phase {m['pid']}  |  {m['category']}", color="#aaa", fontsize=9, va="top"))
        status = "BUILD-UP IN PLAY" if ph.f0 <= f <= ph.f1 else ("context" if f < ph.f0 else f"phase over: {ph.outcome}")
        self._add(t.text(0, .64, status, color="#7cfc00" if ph.f0 <= f <= ph.f1 else "#aaa", fontsize=9, weight="bold", va="top"))
        if st is not None:
            nc, no = int(st["opt_cut"].sum()), len(st["opt_cut"])
            self._add(t.text(0, .50, f"pressers: {st['n_press']}    options cut: {nc}/{no}    passes so far: {npass}", color="white", fontsize=9.5, va="top"))
            ft = "n/a" if st["front3"] is None else f"{st['front3']:.0f} m"
            self._add(t.text(0, .36, f"press line (front-3): {ft} from their goal  ->  {block_of(st['front3'])}", color="white", fontsize=9.5, va="top"))
        self._add(t.text(0, .22, f"shape {m['shape_A']} vs {m['shape_B']}  |  marking: {m['style'] if isinstance(m['style'], str) else 'n/a'}", color="#ccc", fontsize=8.5, va="top"))
        self._add(t.text(0, .08, "green = open option   red dotted = cut   yellow ring = presser   dashed orange = press line", color="#888", fontsize=7, va="top"))
        self.fig.canvas.draw()
        return np.asarray(self.fig.canvas.buffer_rgba())[..., :3][..., ::-1].copy()

def fit(img, W_, H_):
    h, w = img.shape[:2]; s = min(W_ / w, H_ / h); nw, nh = int(w * s), int(h * s)
    out = np.zeros((H_, W_, 3), np.uint8); y0, x0 = (H_ - nh) // 2, (W_ - nw) // 2
    out[y0:y0 + nh, x0:x0 + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA); return out

def to_h264(path):
    if shutil.which("ffmpeg") is None: return path
    tmp = path.with_name(path.stem + "_h264.mp4")
    ok = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(path), "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-crf", "22", str(tmp)]).returncode == 0
    if ok: path.unlink(); tmp.rename(path)
    return path

def render_clip(row, FR, pas, cap, vid_w, vid_h):
    A, B = int(row.team), [t for t in TEAMS if t != int(row.team)][0]
    meta = dict(A=TEAM_NAME[A], B=TEAM_NAME[B], clock=row.clock, category=row.category, pid=int(row.phase_id),
                shape_A=row.shape_A, shape_B=row.shape_B, style=row.press_style)
    panel = Panel(FR, row, pas, meta)
    out = CLIP_DIR / f"{row.name}.mp4"
    ow = {"both": 1280 + PANEL_W, "overlay": 1280, "tactical": PANEL_W}[RENDER_MODE]
    wr = cv2.VideoWriter(str(out), cv2.VideoWriter_fourcc(*"mp4v"), fps / OUT_STRIDE, (ow, PANEL_H))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(row.fs) + VIDEO_FRAME_OFFSET); key_frame = None
    for f in range(int(row.fs), int(row.fe) + 1):
        ok, img = cap.read()
        if not ok: break
        if (f - row.fs) % OUT_STRIDE: continue
        st = FR.get(f)
        if RENDER_MODE != "tactical": img = draw_overlay(img, st, f, meta, row, None, None)
        parts = []
        if RENDER_MODE in ("both", "overlay"): parts.append(fit(img, 1280, PANEL_H))
        if RENDER_MODE in ("both", "tactical"): parts.append(panel.render(f))
        fr_ = np.hstack(parts); wr.write(fr_)
        if st is not None and st["n_press"] >= 1 and key_frame is None and f >= row.f0: key_frame = fr_.copy()
    wr.release(); to_h264(out)
    if key_frame is not None: cv2.imwrite(str(CLIP_DIR / f"{row.name}.jpg"), key_frame)
    return out, key_frame


## 6. Render

In [ ]:
# ===== 6. Render the selected clips =====
CLIP_DIR.mkdir(exist_ok=True, parents=True)
cap = cv2.VideoCapture(str(VIDEO_PATH)); assert cap.isOpened(), f"cannot open {VIDEO_PATH}"
vid_w, vid_h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"video {vid_w}x{vid_h} @ {cap.get(cv2.CAP_PROP_FPS):.2f} fps, {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))} frames (table: {FMAX + 1} frames @ {fps} fps)")
done = []
for i, row in enumerate(CL.itertuples(), 1):
    t0 = time.time()
    FR = clip_state(int(row.fs), int(row.fe), int(row.team))
    pas = PASS_PH[PASS_PH.phase_id == row.phase_id]
    out, key = render_clip(row, FR, pas, cap, vid_w, vid_h)
    done.append((row.name, out, key)); print(f"[{i}/{len(CL)}] {out.name}  ({time.time() - t0:.0f}s, {(row.fe - row.fs) / fps:.0f}s of play)")
cap.release()
print("\nclips saved in:", CLIP_DIR)

# preview: the first pressed frame of each clip
shown = [(n, k) for n, _, k in done if k is not None]
if shown:
    cols = 2; rows_ = int(np.ceil(len(shown) / cols)); fig, axs = plt.subplots(rows_, cols, figsize=(18, 4.6 * rows_)); axs = np.atleast_1d(axs).ravel()
    for a in axs: a.axis("off")
    for a, (n, k) in zip(axs, shown): a.imshow(k[..., ::-1]); a.set_title(n, fontsize=9)
    plt.tight_layout(); plt.show()
try:
    from IPython.display import Video, display
    display(Video(str(done[0][1]), embed=False, width=960))
except Exception as e:
    print("open the mp4s from the folder above (notebook preview unavailable):", e)
